In [1]:
!unzip -q -o ./data/you-tube-comments-signal-vsosai.zip -d ./data/

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

## Dependencies

In [2]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn

import emoji

from tqdm.cli import tqdm
from tqdm import trange

tqdm.pandas()

import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('names', quiet=True)


True

In [3]:
df_train = pd.read_csv('./data/train.csv')
df_test = pd.read_csv('./data/test.csv')
df_train.head()

,id,comment_text,target
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1
1,167,It's a myth that Neanderthals were stupid. The...,1
2,169,Emma is super white.\nIn other news...,1
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1


## EDA

In [4]:
df_train.shape

(51153, 3)

In [5]:
df_train['target'].value_counts()

target
0    38530
1     7808
2     3125
3     1690
Name: count, dtype: int64

In [6]:
df_train['comment_text'][:10]

0    Eh c kii ce favoritisme lui il a tou led iphon...
1    It's a myth that Neanderthals were stupid. The...
2               Emma is super white.\nIn other news...
3    Chelsea and Aubrey are my favourite on Teen Mo...
4    that mustard colour tho 😻😻 p.s absolutely love...
5    OMG u tagged Dina, never see anyone mention he...
6    when you said you're girl twins would be calle...
7                                 Loved this lousie♥♥♥
8    I'm a man united fan but anyone who can't see ...
9    When Christensen speaks better English than Ro...
Name: comment_text, dtype: object

In [7]:
df_train.isna().sum()

id              0
comment_text    9
target          0
dtype: int64

In [8]:
df_train = df_train.fillna("NA")
df_test = df_test.fillna("NA")

## Solutions

In [9]:
def text_feature_extraction(df: pd.DataFrame, col: str):
    df['emoji_count'] = df[col].progress_apply(lambda x: len(emoji.emoji_list(x)))
    df['happy'] = df[col].progress_apply(lambda x: x.count(':)') + x.count(';)') + x.count('(:') + x.count('(;'))
    df['sad'] = df[col].progress_apply(lambda x: x.count(':(') + x.count(';(') + x.count('):') + x.count(');'))
    df['is_na'] = df[col] == "NA"
    return df

In [10]:
df_train = text_feature_extraction(df_train, 'comment_text')
df_test = text_feature_extraction(df_test, 'comment_text')

100%|██████████| 15050/15050 [00:00<00:00, 1337718.81it/s]


### CatBoost

In [11]:
X = df_train.drop(columns=['id', 'target'])
y = df_train['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
model = CatBoostClassifier(
    iterations=50000,
    learning_rate=0.01,
    eval_metric='MultiClass',
    random_seed=42,
    early_stopping_rounds=1000,
    use_best_model=True,
    thread_count=-1,
    task_type='GPU',
    auto_class_weights='Balanced',
    text_features=['comment_text']
)

model.fit(
   X_train,
   y_train,
   eval_set=(X_val, y_val),
   verbose=1000
)

0:	learn: 1.3757626	test: 1.3736035	best: 1.3736035 (0)	total: 7.57ms	remaining: 6m 18s
1000:	learn: 0.6741123	test: 0.5871296	best: 0.5871296 (1000)	total: 3.83s	remaining: 3m 7s
2000:	learn: 0.6147511	test: 0.5497797	best: 0.5497797 (2000)	total: 7.37s	remaining: 2m 56s
3000:	learn: 0.5744478	test: 0.5290956	best: 0.5290956 (3000)	total: 10.9s	remaining: 2m 50s
4000:	learn: 0.5438920	test: 0.5145723	best: 0.5145723 (4000)	total: 14.4s	remaining: 2m 45s
5000:	learn: 0.5184076	test: 0.5032320	best: 0.5032303 (4999)	total: 18s	remaining: 2m 41s
6000:	learn: 0.4968722	test: 0.4943451	best: 0.4943451 (6000)	total: 21.5s	remaining: 2m 37s
7000:	learn: 0.4774578	test: 0.4866584	best: 0.4866584 (7000)	total: 25.1s	remaining: 2m 34s
8000:	learn: 0.4604637	test: 0.4800477	best: 0.4800477 (8000)	total: 28.6s	remaining: 2m 30s
9000:	learn: 0.4450564	test: 0.4741294	best: 0.4741256 (8998)	total: 32.2s	remaining: 2m 26s
10000:	learn: 0.4308684	test: 0.4692514	best: 0.4692514 (10000)	total: 35.7s	r

CatBoostClassifier(auto_class_weights='Balanced', early_stopping_rounds=1000, eval_metric='MultiClass', iterations=50000, learning_rate=0.01, random_seed=42, task_type='GPU', text_features=['comment_text'], use_best_model=True)

In [13]:
subm = df_test.copy()
subm['target'] = model.predict(df_test.drop(columns=['id']))
subm[['id', 'target']].to_csv("subm.csv", index=False)
subm.head()

,id,comment_text,emoji_count,happy,sad,is_na,target
0,200,Me with Arabic. I can never have a conversatio...,0,0,0,False,2
1,607,if I could subscribe to Dude Perfect a million...,0,0,0,False,1
2,1100,We can fortunately say that we lived in the er...,1,0,0,False,1
3,1137,"Not an easy Classic to cover, you and your ban...",0,0,0,False,1
4,1139,"I really really really love this cover, hope H...",0,0,0,False,1


### Linear Models

#### Data preparing

In [14]:
df_train.head()

,id,comment_text,target,emoji_count,happy,sad,is_na
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1,0,0,0,False
1,167,It's a myth that Neanderthals were stupid. The...,1,0,0,0,False
2,169,Emma is super white.\nIn other news...,1,0,0,0,False
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1,4,0,0,False
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1,2,1,0,False


In [15]:
df_extra = pd.read_csv('./data/extra_unlabeled.csv').dropna()

In [16]:
def text_normalize(text: str):
    text = text.lower()
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = "".join([ch for ch in text if ch not in string.punctuation])
    text = emoji.replace_emoji(text, replace='')

    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    text = ' '.join(words)
    return text
    
text_normalize(df_train['comment_text'][4])

'mustard colour tho ps absolut love video whenev someon mention babi name instantli think sim okayyyyy'

In [17]:
df_train_clean = df_train.copy()
df_test_clean = df_test.copy()
df_extra_clean = df_extra.copy()

# df_train_clean = text_feature_extraction(df_train_clean, 'comment_text')
df_train_clean['comment_text'] = df_train_clean['comment_text'].progress_apply(text_normalize)

# df_test_clean = text_feature_extraction(df_test_clean, 'comment_text')
df_test_clean['comment_text'] = df_test_clean['comment_text'].progress_apply(text_normalize)

# df_extra_clean = text_feature_extraction(df_extra_clean, 'comment_text')
df_extra_clean['comment_text'] = df_extra_clean['comment_text'].progress_apply(text_normalize)

df_train_clean.head()

  1%|          | 404/51153 [00:00<00:12, 4037.41it/s]

100%|██████████| 652233/652233 [02:01<00:00, 5375.90it/s]


,id,comment_text,target,emoji_count,happy,sad,is_na
0,59,eh c kii ce favoritism lui il tou led iphon ap...,1,0,0,0,False
1,167,myth neanderth stupid actual brainpow human ti...,1,0,0,0,False
2,169,emma super whitenin news,1,0,0,0,False
3,372,chelsea aubrey favourit teen mom well especi c...,1,4,0,0,False
4,373,mustard colour tho ps absolut love video whene...,1,2,1,0,False


In [18]:
all_text = df_train_clean['comment_text'].tolist() + df_extra_clean['comment_text'].tolist() + df_test_clean['comment_text'].tolist()
all_text[:10]

['eh c kii ce favoritism lui il tou led iphon appl watch ipad macbook en avanc est gratuit wallah je reflechi une ide poir wow ador je sui rich mdrr sa arrivera qau autr',
 'myth neanderth stupid actual brainpow human time got assimil',
 'emma super whitenin news',
 'chelsea aubrey favourit teen mom well especi cole babi',
 'mustard colour tho ps absolut love video whenev someon mention babi name instantli think sim okayyyyy',
 'omg u tag dina never see anyon mention even though she amaz',
 'said your girl twin would call lyla luci sister liter look like omg name lyla',
 'love lousi',
 'im man unit fan anyon cant see hazard best player prem last season mad',
 'christensen speak better english rooney']

In [19]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), stop_words='english')
tfidf.fit(all_text)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [20]:
X_tfidf = tfidf.transform(df_train_clean['comment_text'])
X_tfidf_test = tfidf.transform(df_test_clean['comment_text'])
y = df_train_clean['target']

X_tfidf_train, X_tfidf_val, y_train, y_val = train_test_split(X_tfidf, y, test_size=0.3, random_state=42, stratify=y)

# svd = TruncatedSVD(n_components=2000, random_state=42)
# X_tfidf_train = svd.fit_transform(X_tfidf_train)
# X_tfidf_val = svd.transform(X_tfidf_val)
# X_tfidf_test = svd.transform(X_tfidf_test)

# sc = StandardScaler()
# X_tfidf_train = sc.fit_transform(X_tfidf_train)
# X_tfidf_val = sc.transform(X_tfidf_val)
# X_tfidf_test = sc.transform(X_tfidf_test)

print(f"TF-IDF матрица: {X_tfidf.shape}")
print(f"Ненулевых элементов: {X_tfidf.nonzero()[0].shape[0] / np.prod(X_tfidf.shape) * 100:.2f}%")

TF-IDF матрица: (51153, 20000)
Ненулевых элементов: 0.03%


In [21]:
df_test.shape, df_test_clean.shape, X_tfidf_test.shape

((15050, 6), (15050, 6), (15050, 20000))

In [22]:
model = LogisticRegression(
    max_iter=10000,
    n_jobs=-1,
    class_weight='balanced',
    random_state=42
)
# model.fit(X_tfidf_train, y_train)

# y_pred = model.predict(X_tfidf_val)
# f1_score(y_val, y_pred, average='macro')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_tfidf, y, cv=cv, scoring='f1_macro', n_jobs=-1)
scores.mean()

np.float64(0.6719993202435123)

In [23]:
# subm = df_test.copy()
# subm['target'] = model.predict(X_tfidf_test)
# subm.head()

# subm[['id', 'target']].to_csv("subm.csv", index=False)

In [24]:
model = LogisticRegression(
    max_iter=1000000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
model.fit(X_tfidf_train, y_train)

y_pred = model.predict(X_tfidf_val)
f1_score(y_val, y_pred, average='macro')

0.6673475445853532

In [25]:
subm = df_test.copy()
subm['target'] = model.predict(X_tfidf_test)
subm.head()

subm[['id', 'target']].to_csv("subm.csv", index=False)

In [39]:
model = CatBoostClassifier(
    iterations=10000,
    learning_rate=0.01,
    eval_metric='MultiClass',
    random_seed=42,
    early_stopping_rounds=100,
    use_best_model=True,
    thread_count=-1,
    auto_class_weights='Balanced',
    task_type="GPU"
)

model.fit(
    X_tfidf_train,
    y_train,
    eval_set=(X_tfidf_val, y_val),
    verbose=100
)

0:	learn: 1.3826116	test: 1.3828365	best: 1.3828365 (0)	total: 27ms	remaining: 4m 29s
100:	learn: 1.1651951	test: 1.1873408	best: 1.1873408 (100)	total: 2.31s	remaining: 3m 46s
200:	learn: 1.0550888	test: 1.0966847	best: 1.0966847 (200)	total: 4.53s	remaining: 3m 40s
300:	learn: 0.9805367	test: 1.0363556	best: 1.0363556 (300)	total: 6.72s	remaining: 3m 36s
400:	learn: 0.9222799	test: 0.9891014	best: 0.9891014 (400)	total: 8.9s	remaining: 3m 32s
500:	learn: 0.8753346	test: 0.9518569	best: 0.9518569 (500)	total: 11.1s	remaining: 3m 29s
600:	learn: 0.8357124	test: 0.9202472	best: 0.9202472 (600)	total: 13.2s	remaining: 3m 26s
700:	learn: 0.8007054	test: 0.8931313	best: 0.8931313 (700)	total: 15.4s	remaining: 3m 23s
800:	learn: 0.7704980	test: 0.8692652	best: 0.8692652 (800)	total: 17.5s	remaining: 3m 20s
900:	learn: 0.7435356	test: 0.8479116	best: 0.8479116 (900)	total: 19.6s	remaining: 3m 18s
1000:	learn: 0.7197629	test: 0.8294921	best: 0.8294921 (1000)	total: 21.7s	remaining: 3m 15s
110

CatBoostClassifier(auto_class_weights='Balanced', early_stopping_rounds=100, eval_metric='MultiClass', iterations=10000, learning_rate=0.01, random_seed=42, task_type='GPU', use_best_model=True)

In [40]:
subm = df_test.copy()
subm['target'] = model.predict(X_tfidf_test)
subm.head()

subm[['id', 'target']].to_csv("subm.csv", index=False)